# 목표

연말정산 신고 안내 문서 활용 RAG 시스템 구현

In [138]:
# 기본 경로 설정
# ===============================================================
import os
import requests

PROJECT_NUM = "14"

ROOT_DIR = os.getcwd()


try:
    from google.colab import drive, userdata
    IS_COLAB_MODE = True
    print("코랩 모드")

except ModuleNotFoundError as e:
    IS_COLAB_MODE = False
    ROOT_DIR = os.path.abspath(os.path.join(ROOT_DIR, ".."))
    os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"로컬 모드")


DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(DATA_DIR, exist_ok=True)


# 환경변수 로드 설정
def get_secret(key_name: str):
    if IS_COLAB_MODE:
        return userdata.get(key_name)
    else:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=os.path.join(ROOT_DIR, ".env"))
        return os.getenv(key_name)


if IS_COLAB_MODE:
    import subprocess

    drive.mount('/content/drive')

    # 압축 파일 확보
    if "raw.tar.gz" not in os.listdir():

        headers = {
            "Authorization": f"token {get_secret('GITHUB_PAT')}",
            "Accept": "application/vnd.github.v3+json"
        }

        release_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/tags/data-v{PROJECT_NUM}"

        response = requests.get(release_url, headers=headers)
        asset_id = response.json().get('assets', [])[0]["id"]

        download_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/assets/{asset_id}"
        download_headers = headers.copy()
        download_headers["Accept"] = "application/octet-stream"

        with requests.get(download_url, headers=download_headers, stream=True) as r:
            r.raise_for_status()
            with open("raw.tar.gz", 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

    print("· 압축 파일 있음")


    # 압축 해제 및 경로 처리
    if not os.path.exists(RAW_DIR):
        subprocess.run(["tar", "-xzvf", "raw.tar.gz", "-C", DATA_DIR], check=True)

    print("· 압축 해제 완료")
    print("· 환경 세팅 완료")


    DRIVE_DIR = os.path.join(ROOT_DIR, "drive", "MyDrive")
    SAVE_DIR = os.path.join(DRIVE_DIR, "runs", PROJECT_NUM)

    os.makedirs(SAVE_DIR, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "2024년+원천징수의무자를+위한+연말정산+신고안내.pdf")

로컬 모드


In [214]:
import re
import pandas as pd

import pdfplumber
from img2table.document import PDF
from img2table.ocr import TesseractOCR

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

### 텍스트 데이터

In [ ]:
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

for text in docs:
    text.metadata = {"page": text.metadata["page"]}

In [215]:
# 문서 구조 기반 페이지 정리
DOCS_ABSTRACT = [docs[i] for i in range(2, 10)]
DOCS_INDEX = [docs[i] for i in range(10, 15)]
DOCS_REVISED = [docs[i] for i in range(16, 37)]
DOCS_CONCRETE = docs[38:]

### 요약 페이지(abstract)

In [ ]:
# 메타데이터에 챕터 타이틀 심기
with pdfplumber.open(PDF_PATH) as pdf:
    for page in DOCS_ABSTRACT:
        page_num = page.metadata["page"] 

        head = pdf.pages[page_num].within_bbox((0, 0, 538, 130))
        tail = pdf.pages[page_num].within_bbox((0, 131, 538, 737))

        if head.extract_text():
            target_text = head.extract_text().replace("\n", " ")

        page.metadata["chapter_title"] = target_text
        page.page_content = tail.extract_text()

### 목차 페이지(index)

In [ ]:
for page in DOCS_INDEX[:-1]:
    split = page.page_content.split("\n")

    index_list = [sentence for sentence in split if "·" in sentence]
    
    page.page_content = "\n".join(index_list)


chapter_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=0,
    separators=[r"\n(?=[가-힣])"],
    is_separator_regex=True,
    keep_separator=True
)

DOCS_INDEX = chapter_splitter.split_documents(DOCS_INDEX)

for chapter in DOCS_INDEX:
    split = chapter.page_content.strip().split("\n")

    split[0] = re.sub(r"[\s·.]{2,}\d+", "", split[0])

    chapter.page_content = "\n".join(split)
    chapter.metadata["chapter_title"] = split[0]
    print(chapter.page_content)
    print("="*50)

2024년 귀속 연말정산 개정세법 요약
간소화서비스 전면 개편
1. 주요 개선 내용 ···························································24
2. 소득금액 100만원(총급여 500만원) 산출 방법 ···········24
3. 주의사항 ·····································································24
2024년 귀속 연말정산 주요 일정 ·······························25
1. 회사의 연말정산 업무 일정 ·········································26
2. 원천징수의무자의 서류제출 의무 ································30
원천징수의무자의 연말정산 중점 확인사항
1. 근로소득 원천징수 중점 확인사항(연말정산 이전) ·······34
2. 소득·세액공제 증명서류 중점 확인사항(연말정산 시) ··36
3. 연말정산 과다공제 주요 항목 ······································37
4. 잘못된 소득·세액공제에 따른 가산세 부담 ··················44
근로소득
1. 근로소득의 범위(소법 §20) ·········································48
2. 비과세 근로소득 등 ·····················································52
3. 일용근로소득과 일반근로소득의 구분 ·························73
4. 근로소득의 수입시기(소령 §49) ··································75
5. 근로소득 수입금액 계산 ··············································76
근로소득 원천징수 및 연말정산
1. 근로소득 원천징수 의무 ···········································

### 개정안 페이지(revised)

In [ ]:
# 개정안 항목별로 쪼개기
chapter_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    separators=[r"\d+\s+.*?\n\(.*"],
    is_separator_regex=True,
    keep_separator=True
)

DOCS_REVISED = chapter_splitter.split_documents(DOCS_REVISED)

In [ ]:
# 메타데이터에 표 제목 심기
del_list = list()
for chapter in DOCS_REVISED:
    if chapter.page_content == "원천징수의무자를 위한 \n2024년 연말정산 신고안내":
        del_list.append(chapter)
    elif chapter.page_content == "01. 2024년 귀속 연말정산 개정세법 요약":
        del_list.append(chapter)

    else:
        match = re.search(r"^\d+\s+(.*?)\n\(.*", chapter.page_content, re.DOTALL)
        if match:
            chapter_title = match.group(1).strip()
            chapter.metadata["chapter_title"] = chapter_title
            chapter.page_content = chapter.page_content.replace(chapter_title, "")

for del_ in del_list:
    chapters.remove(del_)

### 표 데이터

In [ ]:

pdf = PDF(
    PDF_PATH, 
    detect_rotation=False,
    pdf_text_extraction=True
)

ocr = TesseractOCR(n_threads=1, lang="eng")

TABLES_BY_PAGE = pdf.extract_tables(
    ocr=ocr,
    implicit_rows=False,
    implicit_columns=False,
    borderless_tables=False,
    min_confidence=40
)

tesseract 5.5.2
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.54 : libtiff 4.7.1 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.5 zlib/1.2.12 liblzma/5.8.2 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.1 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.67.1


In [31]:
# 표 데이터 - 메타데이터 title 보정

TABLE_TITLE_LIST = list()
for page_num, tables in TABLES_BY_PAGE.items():
    for table in tables:
        if table and table.title:
            if len(table.title) > 25:
                table.title = None
            else:
                TABLE_TITLE_LIST.append(table.title)

In [ ]:
# 표 데이터 metadata에 merge

def trim_table(df: pd.DataFrame):
    df.columns = df.iloc[0]
    df = df[1:]
    df.reset_index(drop=True, inplace=True)

    return df


to_delete_list = list()

for page_num, tables in TABLES_BY_PAGE.items():
    if tables:
        docs[page_num].metadata["table"] = {
            f"{page_num}.{i}": trim_table(table.df) for i, table in enumerate(tables)
        }

        for table in tables:
            table = trim_table(table.df)

    else:
        docs[page_num].metadata["table"] = None
        to_delete_list.append(page_num)

for page_num in to_delete_list:
    del TABLES_BY_PAGE[page_num]

In [33]:
TABLES_BY_PAGE[18][0].title

pdf 육안 확인 + df로 바꿔보니 여간 복잡한 게 아니다.

1. 병합된 셀이 많다.
2. 특수문자(O) 같은 게 씹히는 경우가 있다.
3. 타이틀을 일일이 달아줘야 할 것 같다.
4. 표에서 확인할 수 있는 정보는 표를 참고하라고 따로 명령해야 할 듯.

In [34]:
"https://huggingface.co/microsoft/tapex-base-finetuned-wikisql"

'https://huggingface.co/microsoft/tapex-base-finetuned-wikisql'

## RAG 설계

0. 문장 데이터와 표 데이터로 나눈다.
1. 문장 데이터는 OPENAI 모델이 진행.
2. 표 데이터는 https://huggingface.co/microsoft/tapex-base-finetuned-wikisql
3. 허위 정보를 알려줘서는 안 되니, 모르는 건 모른다고 하자.
4. 참고한 페이지 정보를 같이 출력해주어 유저가 더블 체크할 수 있도록 하자.